# A.3: Prose versus code

Compare compression on Shakespeare and on Python source.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Compression as a function of vocabulary size
from collections import Counter

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

text = open("shakespeare.txt", encoding="utf-8").read()

split = int(0.9 * len(text))

train, held_out = text[:split], text[split:]

with open("train.txt", "w", encoding="utf-8") as f:
    f.write(train)

n_bytes = len(held_out.encode("utf-8"))

for V in [512, 1000, 2000, 4000, 8000, 16000]:
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=V, show_progress=False,
                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train(["train.txt"], trainer)
    counts = Counter(tok.encode(train).ids)
    rare = sum(1 for i in range(tok.get_vocab_size()) if counts[i] < 10)
    n_tokens = len(tok.encode(held_out).ids)
    print(V, n_tokens, round(n_bytes / n_tokens, 2), rare)

# From A.2 Token counts across languages
import tiktoken

sentences = {
    "English": "The weather is nice today, so I want to go for a walk in the park.",
    "French": "Il fait beau aujourd'hui, alors je veux aller me promener dans le parc.",
    "Spanish": "Hoy hace buen tiempo, así que quiero ir a pasear por el parque.",
    "German": "Heute ist schönes Wetter, deshalb möchte ich im Park spazieren gehen.",
    "Russian": "Сегодня хорошая погода, поэтому я хочу погулять в парке.",
    "Chinese": "今天天气很好，所以我想去公园散步。",
    "Japanese": "今日は天気がいいので、公園を散歩したいです。",
    "Korean": "오늘은 날씨가 좋아서 공원에 산책하러 가고 싶어요.",
    "Hindi": "आज मौसम अच्छा है, इसलिए मैं पार्क में टहलने जाना चाहता हूँ।",
    "Arabic": "الطقس جميل اليوم، لذلك أريد أن أتمشى في الحديقة.",
}

encs = {n: tiktoken.get_encoding(n) for n in ["gpt2", "cl100k_base", "o200k_base"]}

base = {n: len(e.encode(sentences["English"])) for n, e in encs.items()}

for lang, s in sentences.items():
    row = f"{lang:9}{len(s.encode()):>5} bytes"
    for n, e in encs.items():
        k = len(e.encode(s))
        row += f"{k:>5} ({k / base[n]:.1f}x)"
    print(row)


In [ ]:
held = held_out
code = open("bpe.py", encoding="utf-8").read()
for label, t in [("Shakespeare", held), ("Python", code)]:
    b = len(t.encode("utf-8"))
    print(label, b, {n: round(b / len(e.encode(t)), 2) for n, e in encs.items()})


**Expected output**

Output:

```text
Shakespeare 111540 {'gpt2': 3.09, 'cl100k_base': 3.54, 'o200k_base': 3.6}
Python 3411 {'gpt2': 2.02, 'cl100k_base': 3.98, 'o200k_base': 4.0}
```
